# Sepia iNaturalist: anotación manual y modelos (A1 + A2)

Notebook para revisar con Martin. Solo lee los datos de Drive; **no llama a ningún modelo y no genera archivos**.

Usa dos corridas, cada una en su carpeta:
- **A1** (`pilot_a1`): anotación manual **a ciegas** (sin ver los modelos). Es la referencia válida para medir los prompts.
- **A2** (`a2_test`): anotación **prellenada desde la revisión asistida** (se vieron los modelos antes de decidir). Sirve para entrenar y como referencia, no para medir los prompts.

Las dos se juntan solo en memoria para mirar las fotos. Las métricas van por separado.

1. Interfaz de anotación
2. Resultados de los 4 prompts y de la **moda de los 3 mejores**: métricas y matrices de confusión (A1 y A2 por separado), con una explicación de cada métrica
3. Todas las fotos, una a una
4. Fotos donde al menos un prompt difiere de la anotación manual
5. Fotos donde los 4 prompts coinciden entre sí y difieren de la anotación manual
6. Fotos con notas para preguntar a Martin

En las vistas 4 y 5, `unclear` cuenta siempre como diferencia (en gris en la tabla). En las métricas de la sección 2, `unclear` del modelo es una abstención.

In [ ]:
# --- Preparacion ---
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline
!pip install -q jupyter-bbox-widget            # recuadros en la interfaz de anotacion

sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                      # usar siempre el codigo recien bajado
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia
from sepia.annotate import AnnotationApp
from sepia.disagree import (DisagreementApp, classification_metrics, plot_confusion,
                            count_agreement, rank_versions)

VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'
cfg_a1 = sepia.load_config('configs/pilot_a1.yaml')    # anotacion a ciegas
cfg_a2 = sepia.load_config('configs/a2_test.yaml')     # anotacion asistida
RUNS = [cfg_a1, cfg_a2]

for name, c in (('A1', cfg_a1), ('A2', cfg_a2)):
    obs, pho = sepia.load_tables(c)
    print(f"{name}: {len(obs)} registros, {len(pho)} fotos")
    for v in VERSIONS:
        print(f"   {v}: {len(sepia.load_photo_traits(c, v))} fotos con resultado")

## 1. Interfaz de anotación

Anotación manual por foto e individuo (A1). **Guarda de verdad** en `annotations/manual_HZ.jsonl` (cada guardado agrega una línea; vale la última por registro + foto + letra). Para mostrarla sin cambiar nada: navegar sin apretar *Save*.

In [ ]:
AnnotationApp(cfg_a1, ANNOTATOR).show()

## 2. Resultados de los 4 prompts y de la moda

Cada corrida por separado, sobre las mismas fotos para todos. Lado, erosión y calidad: solo fotos con 1 individuo anotado.

**Moda de los 3 mejores (`mode`)**: para cada foto y atributo se toma el valor que dan al menos 2 de los 3 mejores prompts (votación por mayoría).
- Si los 3 dicen algo distinto, la moda queda en `unclear` (se abstiene). En el conteo también se usa la moda; si no hay (los 3 cuentan distinto), la foto se salta.
- Si un prompt no tiene valor (por ejemplo, vio 2 huesos), votan los otros dos; si no coinciden, `unclear`.
- Los 3 mejores se eligen entre los 4 prompts con el ranking de **A1** (anotación a ciegas): promedio del acuerdo en conteo, lado, erosión y calidad.
- Cuidado: elegirlos y evaluarlos con las mismas fotos de A1 da una cifra algo optimista. Para el paper conviene confirmarla con fotos nuevas anotadas a ciegas.

In [ ]:
ranking, _ = rank_versions(cfg_a1, VERSIONS, n_top=3, save=False)
TOP3 = list(ranking.index[:3])
print('3 mejores (A1):', TOP3)

### Qué mide cada métrica

Con dos clases (por ejemplo, dorsal y ventral) y mirando la clase **dorsal**:

| | el modelo dice dorsal | el modelo dice ventral o `unclear` |
|---|---|---|
| **anotado dorsal** | VP (verdadero positivo) | FN (falso negativo) |
| **anotado ventral** | FP (falso positivo) | VN (verdadero negativo) |

- **Exactitud (accuracy)** = fotos acertadas / todas las fotos. Es la cifra global, pero engaña si una clase es mucho más común que la otra. Aquí un `unclear` del modelo cuenta como error; **accuracy_when_decided** la calcula solo sobre las fotos donde el modelo decidió.
- **Precisión (precision)** de dorsal = VP / (VP + FP). Pregunta: cuando el modelo dice "dorsal", ¿cuántas veces es dorsal? Baja = falsas alarmas.
- **Recall (sensibilidad)** de dorsal = VP / (VP + FN). Pregunta: de los dorsales anotados, ¿cuántos encontró el modelo? Baja = se le escapan.
- **F1** = 2 · precisión · recall / (precisión + recall). Media armónica: solo es alta si las dos son altas.
- **Cobertura (coverage)**: fracción de fotos donde el modelo decidió (no `unclear`). Más `unclear` baja la cobertura y suele subir el acierto cuando decide.
- **macro F1** y **exactitud balanceada** (promedio de los recall): cada clase pesa lo mismo, aunque haya pocas fotos de una de ellas.

Ejemplo práctico (sección 2a): la celda que sigue a las matrices toma la matriz del lado de la moda en A1 y calcula cada métrica paso a paso con sus números.

### 2a. A1: anotación a ciegas (referencia para medir los prompts)

In [ ]:
count_agreement(cfg_a1, VERSIONS, mode_of=TOP3);
summary_a1, per_class_a1, mats_a1 = classification_metrics(cfg_a1, VERSIONS, save=False, mode_of=TOP3)

In [ ]:
plot_confusion(mats_a1, 'side');           # fila 'unclear' = anotado unclear (fuera de las metricas)
plot_confusion(mats_a1, 'erosion');
plot_confusion(mats_a1, 'photo_quality');

#### Ejemplo práctico: las métricas del lado, paso a paso (moda, A1)

In [ ]:
cm = mats_a1[('mode', 'side')]
print(cm, '\n')
rows = [c for c in ('dorsal', 'ventral') if c in cm.index]
n = int(cm.loc[rows].values.sum())                       # fotos con lado anotado dorsal/ventral
hits = sum(int(cm.loc[c, c]) for c in rows if c in cm.columns)
decided = int(cm.loc[rows, [c for c in rows if c in cm.columns]].values.sum())
print(f"Exactitud = aciertos / fotos = {hits} / {n} = {hits / n:.2f}   ('unclear' cuenta como error)")
print(f"Exactitud al decidir = {hits} / {decided} = {hits / decided:.2f}")
print(f"Cobertura = decididas / fotos = {decided} / {n} = {decided / n:.2f}\n")
for c in rows:
    vp = int(cm.loc[c, c]) if c in cm.columns else 0
    fp = int(cm.loc[[r for r in rows if r != c], c].sum()) if c in cm.columns else 0
    fn = int(cm.loc[c].sum()) - vp
    prec = vp / (vp + fp) if vp + fp else float('nan')
    rec = vp / (vp + fn) if vp + fn else float('nan')
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else float('nan')
    print(f"{c}: VP={vp} FP={fp} FN={fn}")
    print(f"   precision = VP/(VP+FP) = {vp}/{vp + fp} = {prec:.2f}")
    print(f"   recall    = VP/(VP+FN) = {vp}/{vp + fn} = {rec:.2f}")
    print(f"   F1        = 2·P·R/(P+R) = {f1:.2f}")

### 2b. A2: anotación asistida (solo referencia)

El acuerdo con los prompts sale **inflado**: la anotación partió de sus respuestas.

In [ ]:
count_agreement(cfg_a2, VERSIONS, mode_of=TOP3);
summary_a2, per_class_a2, mats_a2 = classification_metrics(cfg_a2, VERSIONS, save=False, mode_of=TOP3)

In [ ]:
plot_confusion(mats_a2, 'side');
plot_confusion(mats_a2, 'erosion');
plot_confusion(mats_a2, 'photo_quality');

## 3. Todas las fotos, una a una (A1 + A2)

Foto grande, miniaturas del registro (clic = ver en grande), tabla con los 4 prompts y la anotación manual, evidencia de cada prompt y resumen del registro. Arriba de cada foto se ve la corrida y si la anotación fue **a ciegas** (verde) o **asistida** (naranjo). *Run* filtra por corrida; *Field* por criterio; *Go to ID* salta a un registro.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='all', read_only=True, mode_of=TOP3).show()

## 4. Al menos un prompt difiere de la anotación manual

En cualquier criterio (o en el que se elija en *Field*). Rojo = el prompt decidió otra cosa; gris = el prompt dijo `unclear`.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='any_model_differs', read_only=True, mode_of=TOP3).show()

## 5. Los 4 prompts coinciden entre sí y difieren de la anotación manual

Candidatas más fuertes a error de anotación, o a un criterio distinto entre la persona y los prompts. Incluye los casos donde los 4 dicen `unclear` y la persona decidió, y donde la persona dijo `unclear` y los 4 decidieron lo mismo.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='all_models_agree_differ', read_only=True, mode_of=TOP3).show()

## 6. Preguntas para Martin

Fotos donde alguna nota de la anotación menciona a Martin (martin, Martín, marthin). La nota aparece destacada arriba.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='ask_martin', read_only=True, mode_of=TOP3).show()